In [5]:
import nltk
import numpy as np
import networkx as nx
import pandas as pd
from nltk.tokenize import sent_tokenize, word_tokenize
import re
from collections import Counter
import joblib
from tqdm import tqdm
import os

# Download the Punkt tokeniser models
nltk.download('punkt')
# Download part-of-speech (POS) tagging model
nltk.download('averaged_perceptron_tagger')

# Load the preprocessed data for keyword analysis
df = pd.read_pickle("data_preprocessed.pkl")

# Verify the number of rows
print("Number of rows:", df.shape[0])

# Verify the number of columns
print("Columns:", df.columns.tolist())

# Ensure that analysis is based on the abstract column
text_column = 'abstract'

# Regex for valid words
Valid_word = re.compile(r'^[a-zA-Z\-]+$')

#Calculate universal IDF to measure word importance for all of the abstracts
def calculate_universal_IDF(preprocessed_tokens_list):
    N = len(preprocessed_tokens_list)
    document_frequency = Counter()
    for tokens in tqdm(preprocessed_tokens_list, desc="IDF Progress"):
        unique_words = set(tokens)
        document_frequency.update(unique_words)
    idf = {w: np.log(N / (1 + f)) for w, f in document_frequency.items()}
    return idf
preprocessed_tokens_list = [str(text).split() for text in df[text_column].tolist()]
universal_IDF = calculate_universal_IDF(preprocessed_tokens_list)

#convert a sentence into a numerical vector using TF-IDF
def sentence_vector(tokens, vocabulary, universal_IDF):
    if not tokens:
        return np.zeros(len(vocabulary))
    vector = np.zeros(len(vocabulary))
    Term_frequency = Counter(tokens)
    for word, count in Term_frequency.items():
        if word in vocabulary:
            vector[vocabulary[word]] = (count / len(tokens)) * universal_IDF.get(word, 1.0)
    return vector

#Turn text it into sentences and a numerical vocabulary so that TextRank can calculate sentence importance
def textrank_summarise(preprocessed_text, top_n=3, position_bias=0.3):
    if not isinstance(preprocessed_text, str):
        return preprocessed_text
    sentences = sent_tokenize(preprocessed_text)
    if len(sentences) <= top_n:
        return preprocessed_text
    all_words = set(preprocessed_text.split())
    vocabulary = {w: i for i, w in enumerate(all_words)}

#Convert all sentences into normalised numerical vectors so they can be fairly based on their similarities
    vectors = np.array([sentence_vector(sent.split(), vocabulary, universal_IDF) for sent in sentences])
    #Normalise each sentence vector to have unit length
    norm = np.linalg.norm(vectors, axis=1, keepdims=True)
    norm[norm == 0] = 1e-10
    vectors = vectors / norm
    #Compute cosine similarity between all pairs of sentences
    similarity_matrix = np.clip(vectors @ vectors.T, 0, None)
    #Remove self-loops in the similarity graph
    np.fill_diagonal(similarity_matrix, 0)
    #Convert the similarity matrix into a graph using NetworkX
    Graph_sentences = nx.from_numpy_array(similarity_matrix)
    #Apply PageRank algorithm to the graph
    scores = nx.pagerank(Graph_sentences, max_iter=200, tol=1e-6)

    #Assign higher raw scores to sentences that appear earlier
    position_raw = np.exp(-0.5 * np.arange(len(sentences)))
    #Normalise the raw position scores so they sum to 1
    position_score = position_raw / position_raw.sum()
    #Combine TextRank score and position score into a single importance score for each sentence
    final_scores = {i: (1 - position_bias) * scores[i] + position_bias * position_score[i] for i in range(len(sentences))}
    #Pick the top important sentences based on the combined score
    ranked_index_row = sorted(final_scores, key=final_scores.get, reverse=True)[:top_n]
    #Return the selected sentences in the original order to form a readable summary
    return ' '.join(sentences[i] for i in sorted(ranked_index_row))

#use Penn Treebank tags to define the structure of a Noun phrase
Noun_Phrase_Grammar = r'KP: {(<JJ.*>|<NN.*>)*<NN.*>}'
Noun_Phrase_Parser = nltk.RegexpParser(Noun_Phrase_Grammar)

def extract_noun_phrases(text):
    #Split text into individual words
    tokens = word_tokenize(text)
    #Assign grammatical labels
    tagged = nltk.pos_tag(tokens)
    #turn text into structured tree
    tree = Noun_Phrase_Parser.parse(tagged)
#store extracted noun phrases
    phrases = []
    #Loop through noun phrase chunks
    for subtree in tree.subtrees(filter=lambda t: t.label() == 'KP'):
        words = [w.lower() for w, _ in subtree.leaves()]
        valid_words = [w for w in words if Valid_word.match(w)]
        #filter phrase length
        if 1 <= len(valid_words) <= 5:
            #Join words into phrase
            phrase = ' '.join(valid_words)
            #Store phrase
            phrases.append(phrase)
    #output all the extracted noun phrases
    return phrases

def filter_redundant_phrases(ranked_phrases, top_n):
    filtered = []
    #Loop through ranked phrases
    for phrase in ranked_phrases:
        #check redundancy
        if not any(phrase in f or f in phrase for f in filtered):
            filtered.append(phrase)
        if len(filtered) >= top_n:
            break
    #Output final cleaned keyphrases
    return filtered

#extract and rank important keyphrases using TextRank graph
def textrank_keyphrases(text, top_n=10, window=5):
    if not isinstance(text, str):
        return []
    phrases = extract_noun_phrases(text)
    if len(phrases) < 3:
        return phrases[:top_n]
    unique_phrases = list(dict.fromkeys(phrases))
    Graph_sentences = nx.Graph()
    Graph_sentences.add_nodes_from(unique_phrases)
    for i, p in enumerate(phrases):
        for q in phrases[i+1:i+window]:
            if p != q:
                if Graph_sentences.has_edge(p, q):
                    Graph_sentences[p][q]['weight'] += 1
                else:
                    Graph_sentences.add_edge(p, q, weight=1)

    #Apply PageRank
    PageRank = nx.pagerank(Graph_sentences, weight='weight', max_iter=200, tol=1e-6)
    #Define final scoring function
    def final_score(phrase):
        return PageRank.get(phrase, 0) * (1 + 0.15 * (len(phrase.split()) - 1))
    #Rank phrases
    ranked = sorted(unique_phrases, key=final_score, reverse=True)
    #Remove overlapping phrases
    return filter_redundant_phrases(ranked, top_n)

# 4. Parallel Processing
print("\nRunning TextRank summarisation & keyphrase extraction...")

def process_summaries(texts):
    return [textrank_summarise(t) for t in texts]

def process_keyphrases(texts):
    return [textrank_keyphrases(t) for t in texts]

#Split the data into chunks for parallel processing
n_cores = joblib.cpu_count()
text_chunks = np.array_split(df[text_column].astype(str), n_cores)

#Run parallel summarisation
df['summary'] = sum( joblib.Parallel(n_jobs=n_cores)( joblib.delayed(process_summaries)(chunk) for chunk in tqdm(text_chunks, desc="Summarising") ), [] )

#Run parallel keyphrase extraction
df['keyphrases'] = sum( joblib.Parallel(n_jobs=n_cores)( joblib.delayed(process_keyphrases)(chunk) for chunk in tqdm(text_chunks, desc="Extracting Keyphrases") ), [] )

#Displaying the summary and keyphrases for the first 3 rows of abstracts
pd.set_option('display.max_colwidth', 120)
print("\n"+"="*70)
print("First 3 rows")
print("="*70)
for index_row, row in df[[text_column,'summary','keyphrases']].head(3).iterrows():
    print(f"\n[Row {index_row}]")
    print(f"Summary: {str(row['summary'])[:200]}...")
    print(f"Keyphrases: {', '.join(row['keyphrases'])}")
df.to_pickle("data_with_nlp.pkl")
df[[text_column, 'summary', 'keyphrases']].to_csv("nlp_results.csv", index=False)
print("\nSaved > data_with_nlp.pkl, nlp_results.csv")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\clare\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\clare\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


Number of rows: 27121
Columns: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'abstract', 'report-no', 'categories', 'versions', 'year', 'cleaned_abstract', 'period']


IDF Progress: 100%|██████████| 27121/27121 [00:00<00:00, 29378.95it/s]



Running TextRank summarisation & keyphrase extraction...


Extracting Keyphrases: 100%|██████████| 8/8 [00:00<00:00, 3068.25it/s]



First 3 rows

[Row 0]
Summary:   This paper uncovers and explores the close relationship between Monte Carlo
Optimization of a parametrized integral (MCO), Parametric machine-Learning
(PL), and `blackbox' or `oracle'-based optimiza...
Keyphrases: mco, pl techniques, bo, integrand, immediate sampling, contributions, values, new application domain, sample point locations, sample location information

[Row 1]
Summary:   In these notes we formally describe the functionality of Calculating Valid
Domains from the BDD representing the solution space of valid configurations.
The formalization is largely based on the CLa...
Keyphrases: solution space, calculating valid domains, bdd, valid configurations, clab configuration framework, functionality, formalization, notes

[Row 2]
Summary:   Motivation: Profile hidden Markov Models (pHMMs) are a popular and very
useful tool in the detection of the remote homologue protein families. We present HMMER-STRUCT, a model construction algorithm...
Keyphr

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re
import nltk
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('omw-1.4')

#Load data
lemmatizer=WordNetLemmatizer()
df=pd.read_pickle("data_with_nlp.pkl")

#Defines the boundaries of each year range
bins=[2007, 2012, 2017, 2022]
labels=["2007–2011", "2012–2016", "2017–2021"]
df['year_bin']=pd.cut(df['year'], bins=bins, labels=labels, right=False)

# remove plural "s" if it's a simple word ending in s
def clean_phrase(text):
    tokens=text.split()
    lemmas=[lemmatizer.lemmatize(t) for t in tokens]
    cleaned=[]
    for t in lemmas:
        if len(t) > 3 and t.endswith("s") and not t.endswith("ss"):
            t=t[:-1]
        cleaned.append(t)
    return " ".join(cleaned)

#keep phrases that actually represent meaningful AI methods
AI_methods=["support vector machine", "svm", "convolutional neural network", "cnn", "recurrent neural network", "rnn", "transformer", "bert", "self attention", "attention mechanism", "reinforcement learning", "backpropagation", "stochastic gradient descent", "gradient descent", "deep learning" ]

#filter out generic terms
Generic_terms=set(["feature", "features", "accuracy", "prediction", "loss", "classification", "regression", "network", "neural", "nlp", "vision", "learning", "model", "models", "training", "dataset", "datasets", "machine", "optimization", "gradient", "clustering"])

#explode keyphrases
df_exploded=df[['year_bin', 'keyphrases']].explode('keyphrases')
df_exploded=df_exploded.dropna()

#Apply cleaning function
df_exploded['keyphrases']=df_exploded['keyphrases'].apply(clean_phrase)

#keyword validation
def valid_ai_ml(text):
    text=str(text)
    signal_keyword=any(phrase in text for phrase in AI_methods)
    # reject if contains generic words
    tokens=set(text.split())
    noise=any(term in tokens for term in Generic_terms)
    return signal_keyword and not noise

df_exploded=df_exploded[df_exploded['keyphrases'].apply(valid_ai_ml)]

#AI/ML Research Activity
growth_score = (df_exploded.groupby('year_bin').size().reset_index(name='ai_ml_research_activity_score'))

print("\n AI/ML Research Activity Score")
print(growth_score)

#Top Keyphrases per Era
trend=(df_exploded.groupby(['year_bin', 'keyphrases']) .size() .reset_index(name='count').sort_values(['year_bin','count'], ascending=[True, False]) )
top_trends = trend.groupby('year_bin').head(10)

print("\nTop AI/ML Keyphrases per Era")
print(top_trends)

# 11. visualisation of results
plt.figure()
plt.plot(growth_score['year_bin'].astype(str),growth_score['ai_ml_research_activity_score'],marker='o')
plt.xlabel("Year Range")
plt.ylabel("AI/ML Research Activity Score")
plt.title("AI/ML Research Evolution")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()